###**Data Reading**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.dropdown(name='initial_load_flag',defaultValue='1',choices=['1','0'])


In [0]:
df = spark.sql(
    "select * from portfolio_project.silver.customers"
)

In [0]:
df = df.dropDuplicates(subset = ['customer_id'])

#**Marking Old and New records**


In [0]:
initial_load_flag = int(dbutils.widgets.get('initial_load_flag'))

if initial_load_flag == 0:
   
    df_old = spark.sql('''
                       
                       select DimCustomerKey, customer_id,create_date, update_date 
                       from portfolio_project.gold.DimCustomers
                       
                       '''
                       )
    
else:
    df_old = spark.sql('''
                       
                       select 0 DimCustomerKey, 0 customer_id, 0 create_date, 0 update_date 
                       from portfolio_project.silver.customers
                       where 1=0
                       '''
                       )


In [0]:
df_old = df_old.withColumnRenamed('DimCustomerKey', 'old_DimCustomerKey')\
    .withColumnRenamed('customer_id', 'old_customer_id')\
    .withColumnRenamed('create_date', 'old_create_date')\
    .withColumnRenamed('update_date', 'old_update_date')


In [0]:
df_joined = df.join(df_old,df.customer_id == df_old.old_customer_id,how='left')

In [0]:
display(df_joined.limit(10))

customer_id,email,city,state,domains,full_name,DimCustomerKey,old_DimCustomerKey,old_customer_id,old_create_date,old_update_date
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney,1,null,null,null,null
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers,2,null,null,null,null
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes,3,null,null,null,null
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott,4,null,null,null,null
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez,5,null,null,null,null
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,6,null,null,null,null
C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle,7,null,null,null,null
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos,8,null,null,null,null
C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green,9,null,null,null,null
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,10,null,null,null,null


In [0]:
df_new_records = df_joined.filter(df_joined.old_DimCustomerKey.isNull())

In [0]:
df_old_records = df_joined.filter(df_joined.old_DimCustomerKey.isNotNull())


In [0]:
df_old_records = df_old_records.drop('old_DimCustomerKey','old_customer_id','old_update_date')

df_old_records = df_old_records.withColumnRenamed('old_create_date','create_date')
df_old_records = df_old_records.withColumn('create_date', to_timestamp((col("create_date"))))

df_old_records = df_old_records.withColumn('update_date', current_timestamp())

In [0]:
df_old_records.limit(10).display()

customer_id,email,city,state,domains,full_name,create_date,update_date
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,2026-08-23T10:04:56.019Z,2026-08-23T10:09:15.556Z
C00045,dkhan@hotmail.com,North Kara,OK,hotmail.com,Matthew Lee,2026-08-23T10:04:56.019Z,2026-08-23T10:09:15.556Z
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,2026-08-23T10:04:56.019Z,2026-08-23T10:09:15.556Z
C00056,cruzkristen@hotmail.com,Catherineberg,WA,hotmail.com,Tina Cantu,2026-08-23T10:04:56.019Z,2026-08-23T10:09:15.556Z
C00062,kimdennis@yahoo.com,Kaitlynburgh,MI,yahoo.com,Shelley Holland,2026-08-23T10:04:56.019Z,2026-08-23T10:09:15.556Z
C00094,rogersrandall@gonzales.org,Harrisonmouth,AK,gonzales.org,Hannah Olson,2026-08-23T10:04:56.019Z,2026-08-23T10:09:15.556Z
C00104,gloria15@yahoo.com,New Danielle,VA,yahoo.com,Patrick Meadows,2026-08-23T10:04:56.019Z,2026-08-23T10:09:15.556Z
C00105,christine52@kennedy.com,Crystalmouth,AL,kennedy.com,April Wright,2026-08-23T10:04:56.019Z,2026-08-23T10:09:15.556Z
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,2026-08-23T10:04:56.019Z,2026-08-23T10:09:15.556Z
C00152,tara25@guzman-nelson.net,Ashleyside,ID,guzman-nelson.net,David Salazar,2026-08-23T10:04:56.019Z,2026-08-23T10:09:15.556Z


In [0]:
df_new_records = df_new_records.drop('old_DimCustomerKey','old_customer_id','old_update_date','old_create_date')

df_new_records = df_new_records.withColumn('create_date', current_timestamp())
df_new_records = df_new_records.withColumn('update_date', current_timestamp())


In [0]:
if initial_load_flag == 1:
    max_surrodate_key = 0

else:
    df_maxsur = spark.sql('''create max(DimCustomerKey) as max_surrodate_key from portfolio_project.gold.DimCustomers''')
    max_surrodate_key = df_maxsur.collect()[0]['max_surrodate_key']

In [0]:
df_new_records = df_new_records.withColumn("DimCustomerKey", lit(max_surrodate_key)+col('DimCustomerKey'))

In [0]:
df_final = df_new_records.unionByName(df_old_records)
display(df_final.limit(10))

customer_id,email,city,state,domains,full_name,DimCustomerKey,create_date,update_date
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney,1,2026-08-23T09:27:35.188Z,2026-08-23T09:27:35.188Z
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers,2,2026-08-23T09:27:35.188Z,2026-08-23T09:27:35.188Z
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes,3,2026-08-23T09:27:35.188Z,2026-08-23T09:27:35.188Z
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott,4,2026-08-23T09:27:35.188Z,2026-08-23T09:27:35.188Z
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez,5,2026-08-23T09:27:35.188Z,2026-08-23T09:27:35.188Z
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,6,2026-08-23T09:27:35.188Z,2026-08-23T09:27:35.188Z
C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle,7,2026-08-23T09:27:35.188Z,2026-08-23T09:27:35.188Z
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos,8,2026-08-23T09:27:35.188Z,2026-08-23T09:27:35.188Z
C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green,9,2026-08-23T09:27:35.188Z,2026-08-23T09:27:35.188Z
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,10,2026-08-23T09:27:35.188Z,2026-08-23T09:27:35.188Z


In [0]:
if spark.catalog.tableExists("portfolio_project.gold.DimCustomers"):
        
    delta_object = DeltaTable.forPath(spark,"abfss://gold@databrcks.dfs.core.windows.net/DimCustomers")

    delta_object.alias('target').merge(df_final.alias('src'), 'target.DimCustomerKey = src.DimCustomerKey')\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
    
else:

    df_final.write.mode("overwrite")\
        .option('path',"abfss://gold@databrcks.dfs.core.windows.net/DimCustomers")\
        .saveAsTable("portfolio_project.gold.DimCustomers")
